# NB-15 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-15**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-15.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This is the computational bridge from a path’s physical exposure to weighted geometry and scenario risk. The retained controlled operator is not an observed ISO PTDF.

- [Volume II, Chapter 8 — From prices back to the graph](https://sherandoresearch.github.io/power-markets-series/vol2_grid_geometry.html#c8)
- [Volume IV, Chapter 21 — FTRs as a network-derivative laboratory](https://sherandoresearch.github.io/power-markets-series/vol4_testbed_atlas.html#chiv-21)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-15', require_key=True)
print(setup_status)


# NB-15 — FTR magnitude, direction, span, and scenario risk
Companion to Volume II §8 and Volume IV §21. Read the existing Canonical E
fixture in its original node, branch, and path order. No random draws, market
simulation, or new observations occur. Calculations use authenticated private methods.
The operator is controlled, not an ISO PTDF.

We will separate physical exposure, weighted geometry, and realized covariance;
then reproduce the existing signed/absolute screening results at matched budgets.

In [ ]:
NOTEBOOK_ID='NB-15'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

print('Locally inspectable retained support:', inputs['raw_inspection'])
values={}
def run_saved(name,parents=(),request=None):
    entry=inputs['requests'][name]
    values[name]=execute(name,entry['method'],request if request is not None else entry['request'],parents)
    return values[name]
path_results={};path_parents={}
indices=inputs['display_indices_in_complete_universe']
for state in ('intact','outage'):
    chunks=[];parents=[]
    for start in (0,250,500):
        name=f'geometry:{state}:{start}:{min(start+250,650)}'
        result=run_saved(name)['result']['exposure_matrix']
        chunks.append(np.asarray(result['values']))
        if any(start<=i<min(start+250,650) for i in indices):parents.append(name)
    path_results[state]=np.concatenate(chunks,axis=1)[:,indices]
    path_parents[state]=parents
e,e_out=path_results['intact'],path_results['outage']
h=np.asarray(inputs['explanatory_path_matrix']);labels=inputs['display_labels']
print({'H_nodes_by_paths':h.shape,'Psi_branches_by_nodes':(38,26),'E_branches_by_paths':e.shape})
print('E = Psi H = -T; congestion payoff = Tᵀ mu = -Eᵀ mu.')
metrics={};geometries={};geometry_diagnostics={}
def displayed_geometry(output):
    result=output['result'];d=output['diagnostics']['retained_factor_diagnostics']
    return SimpleNamespace(factor=np.asarray(d['factor']),gram=np.asarray(result['gram']),magnitude=np.asarray(result['magnitude']),cosine=np.asarray(result['cosine'],float),valid=np.asarray(result['valid_path_mask'],bool),rank=d['numerical_rank'])
for kind in ('identity','mean_absolute_dual','dual_covariance'):
    metric=run_saved('screen:'+kind)['result']['constraint_metric'];metrics[kind]=np.asarray(metric['values'])
    name='geometry:'+kind;q=copy.deepcopy(inputs['requests'][name]['request'])
    q['inputs']['exposure']['values']=e.tolist();parents=list(path_parents['intact'])
    if kind!='identity':
        assert metric['constraint_ids']==q['inputs']['exposure']['constraint_ids']
        q['inputs']['metric']['values']=np.diag(metrics[kind]).tolist() if kind=='mean_absolute_dual' else metric['values']
        parents.append('screen:'+kind)
    output=run_saved(name,parents,q);geometries[kind]=displayed_geometry(output);geometry_diagnostics[kind]=output['diagnostics']['retained_factor_diagnostics']
w=np.asarray(inputs['original_susceptance'])
name='geometry:coordinate-transform';q=copy.deepcopy(inputs['requests'][name]['request'])
q['inputs']['exposure']['values']=(e/w[:,None]).tolist()
q['inputs']['metric']['values']=(w[:,None]*metrics['dual_covariance']*w[None,:]).tolist()
transformed=displayed_geometry(run_saved(name,path_parents['intact']+['screen:dual_covariance'],q))
name='geometry:square-root-weighting';q=copy.deepcopy(inputs['requests'][name]['request'])
q['inputs']['exposure']['values']=(np.sqrt(np.diag(metrics['mean_absolute_dual']))[:,None]*e).tolist()
weighted_geometry=displayed_geometry(run_saved(name,path_parents['intact']+['screen:mean_absolute_dual'],q))
name='geometry:cancelling-reversal';q=copy.deepcopy(inputs['requests'][name]['request'])
q['inputs']['exposure']['values']=(e[:,3]+e[:,4])[:,None].tolist()
zero=displayed_geometry(run_saved(name,path_parents['intact'],q))
name='geometry:scenario';q=copy.deepcopy(inputs['requests'][name]['request'])
for row,exposure in zip(q['inputs']['scenarios'],(e,e_out),strict=True):row['exposure']['values']=exposure.tolist()
output=run_saved(name,path_parents['intact']+path_parents['outage'],q)
scenario=displayed_geometry(output);variability=output['diagnostics']['retained_scenario_variability']
mean=np.asarray(variability['mean_exposure']);gap=np.asarray(variability['gram_gap'])
rows=[]
for name,entry in inputs['requests'].items():
    if entry['method']!='budgeted_cosine_screen':continue
    # The retained positive-T scenario factor is a distinct declared artifact.
    # Do not invent a dataflow edge from the reconstructed all-path responses.
    result=run_saved(name)['result']['candidate_selection'];policy=entry['request']['inputs']['selection_policy']
    row=dict(result,budget=policy['budget'],policy=policy['cosine_policy'])
    rows.append({k:row[k] for k in inputs['source_screen_row_fields']})

## Three metrics, three meanings
Identity weights measure unweighted branch-exposure geometry. Mean absolute
duals weight historically expensive constraints; they are not covariance.
The full signed-dual covariance reproduces path payoff covariance under this
fixed operator. Both data-derived metrics use intact hours in the existing
first-250-day training window. Their units differ, so compare cosine and
within-metric norms rather than treating absolute magnitudes as interchangeable.

In [ ]:
print('Runtime factor ranks use the current backend default SVD tolerance. Tiny covariance eigenvalues can change these counts across platforms; geometry identities are checked separately, with no structural-rank claim.')
for name,geo in geometries.items():
    assert geometry_diagnostics[name]['direct_gram_max_absolute_difference']<1e-7
    assert geometry_diagnostics[name]['minimum_gram_eigenvalue']>-1e-7
    print(name,'runtime factor rank:',geo.rank,'first-six magnitudes:',geo.magnitude[:6].round(4))
np.testing.assert_allclose(weighted_geometry.gram,geometries['mean_absolute_dual'].gram,atol=1e-9)
np.testing.assert_allclose(transformed.gram,geometries['dual_covariance'].gram,atol=1e-7)

## A reversal is opposite exposure, not a different magnitude
The existing W3→N2 and N2→W3 pair has cosine −1 and equal norms. Absolute
similarity merges the two directions; a positive-only neighbor rule discards
opposition. Neither policy is automatically the right portfolio constraint.
A cancelling portfolio has zero exposure, whose cosine is undefined.

In [ ]:
geo=geometries['identity'];i,j=3,4
np.testing.assert_allclose(e[:,i],-e[:,j],atol=1e-12)
assert abs(geo.cosine[i,j]+1)<1e-12
assert abs(geo.magnitude[i]-geo.magnitude[j])<1e-12
print({'pair':[labels[i],labels[j]],'signed':geo.cosine[i,j],
       'absolute':abs(geo.cosine[i,j]),'positive_only':max(geo.cosine[i,j],0)})
for policy in ('signed','absolute','positive_only'):
    candidates=[k for k in range(len(labels)) if k!=i and (policy!='positive_only' or geo.cosine[i,k]>0)]
    score=lambda k: abs(geo.cosine[i,k]) if policy=='absolute' else geo.cosine[i,k]
    nearest=sorted(candidates,key=lambda k:(-score(k),k))[:3]
    print(policy,'neighbors of',labels[i],[(labels[k],round(float(geo.cosine[i,k]),4)) for k in nearest])
assert not zero.valid[0] and np.isnan(zero.cosine[0,0])
plt=setup_plot()
fig,axes=plt.subplots(1,2,figsize=(12,4))
im=axes[0].imshow(geo.cosine[:6,:6],vmin=-1,vmax=1,cmap='RdBu')
axes[0].set_xticks(range(6),labels[:6],rotation=45,ha='right');axes[0].set_yticks(range(6),labels[:6])
axes[0].set_title('Signed direction · identity metric');fig.colorbar(im,ax=axes[0])
axes[1].bar(range(6),geo.magnitude[:6],color='#88C0D0')
axes[1].set_xticks(range(6),labels[:6],rotation=45,ha='right');axes[1].set(ylabel='Exposure norm',title='Magnitude is a separate quantity')
fig.tight_layout();plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(11,3.5))
im=ax.imshow(e[:,:6].T,aspect='auto',cmap='RdBu',vmin=-np.abs(e[:,:6]).max(),vmax=np.abs(e[:,:6]).max())
ax.set_yticks(range(6),labels[:6]);ax.set(xlabel='Frozen physical branch row (0–37)',title='Raw E = Psi H exposure · first six paths')
fig.colorbar(im,ax=ax,label='MW flow per MW transfer');fig.tight_layout();plt.show()

## Expected quadratic geometry is not an averaged operator
Reuse the saved research experiment's 75% intact / 25% outage mixture.
These are declared experimental weights, not inferred outage probabilities.
The difference between the two Grams is PSD and measures operator variability.

In [ ]:
assert variability['weighted_difference_identity_error']<1e-10
assert variability['minimum_gap_eigenvalue']>-1e-10
print({'relative_gram_gap':variability['relative_gram_gap'],'scenario_factor_rank':scenario.rank})
for row in rows:print({k:row[k] for k in ('budget','policy','rank','worst_signed_coverage','relative_span_residual')})
fig,axes=plt.subplots(1,2,figsize=(11,4))
for policy,color in [('signed','#88C0D0'),('absolute','#EBCB8B')]:
    selected=[r for r in rows if r['policy']==policy]
    axes[0].plot([r['budget'] for r in selected],[r['worst_signed_coverage'] for r in selected],'-o',color=color,label=policy)
    axes[1].plot([r['budget'] for r in selected],[r['relative_span_residual'] for r in selected],'-o',color=color,label=policy)
for ax in axes:
    ax.set_xlabel('Candidate budget');ax.legend(facecolor='#3B4252',labelcolor='#ECEFF4')
axes[0].set(ylabel='Worst directed cosine coverage',title='Direction coverage')
axes[1].set(ylabel='Relative span residual',title='Linear span coverage')
fig.tight_layout();plt.show()

## Existing decision comparison: preserve the mixed result
These rows are read from the retained experiment, not regenerated. The
experiment uses the same simulated return samples and budgets across policies,
with long-only mean–variance quantities. It is not an auction replay.
Positive differences favor signed screening; negative differences favor
absolute screening. Monte Carlo SE describes these existing trials only.

In [ ]:
paired=inputs['retained_decision_summary']
assert inputs['retained_decision_summary_only']
print('Historical retained decision summaries; no samples regenerated and no accepted scientific Claim.')
for row in paired:print(row)
assert any(r['signed_minus_absolute_utility']<0 for r in paired)
assert any(r['signed_minus_absolute_utility']>0 for r in paired)

In [ ]:
for control in inputs['negative_controls']:reject(control['method'],control['request'])
checked=dict(screen_rows=rows,paired_decision_results=paired,scenario_gram_gap_relative=variability['relative_gram_gap'],historical_paired_results_recomputed=False,scientific_acceptance=False,new_synthetic_data=False)
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(full_650_path_support=True,full_6000_hour_metric_support=True,source_order_ties=True,source_screen_parity=True,legacy_geometry_unchanged=True,original_invalid_domains_refused=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)

## Exercise and boundary
Inspect budget 48 in the saved matched and outage-shift comparisons. Explain
why equal span coverage does not guarantee equal directional coverage or equal
utility. The saved utility differences change sign across those regimes.
Geometry alone does not establish forecast skill, funded FTR cash returns,
or universal superiority of either screening policy.